# 0. Problem
## 180. Consecutive Numbers — Medium
Return numbers that appear at least three rows in a row when ordered by id.
Official: https://leetcode.com/problems/consecutive-numbers/

# 1. Setup

In [ ]:
import pandas as pd
logs_rows=[(1,1),(2,1),(3,1),(4,2),(5,1),(6,2),(7,2),(8,2)]; logs_pd=pd.DataFrame(logs_rows,columns=["id","num"]); logs_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate(); logs_spark=spark.createDataFrame(logs_rows,["id","num"]); logs_spark.createOrReplaceTempView("Logs")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""WITH x AS (SELECT id,num,LAG(num,1) OVER(ORDER BY id) AS p1,LAG(num,2) OVER(ORDER BY id) AS p2 FROM Logs) SELECT DISTINCT num AS ConsecutiveNums FROM x WHERE num=p1 AND num=p2 ORDER BY ConsecutiveNums"""); sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
ordered=logs_pd.sort_values("id").copy(); mask=ordered["num"].eq(ordered["num"].shift(1))&ordered["num"].eq(ordered["num"].shift(2)); result_pd=ordered.loc[mask,["num"]].drop_duplicates().rename(columns={"num":"ConsecutiveNums"}).sort_values("ConsecutiveNums").reset_index(drop=True); result_pd

# 4. PySpark Solution

In [ ]:
w=Window.orderBy("id"); result_spark=(logs_spark.withColumn("p1",F.lag("num",1).over(w)).withColumn("p2",F.lag("num",2).over(w)).filter((F.col("num")==F.col("p1"))&(F.col("num")==F.col("p2"))).select(F.col("num").alias("ConsecutiveNums")).distinct().orderBy("ConsecutiveNums")); result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| previous rows | `LAG()` | `.shift()` | `Window + lag()` |
| deduplicate answer | `DISTINCT` | `.drop_duplicates()` | `.distinct()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Logs

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: logs_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: logs_spark